## Modelling of pion decay 

THe production of $\mu+$ from SM events is proceeded via four channels namely, CC_numu, CC_nuebar, NC_numu, NC_nuebar where CC and NC stands for charged and neutral current respectively. During hadronization,light mesons such as pions and kaons are also produced. These light mesons can further decay to $\mu^+$ + $\nu_{\mu}$. So modelling such events are important to study the effect it has on the background.

EXTRA NOTE: I have created an event file with only one event to study the code. Also, currently I have only included pion decay ( will add kaon once everything is confirmed)

In [1]:
###Brand new with weight
import pandas as pd
import numpy as np
import math
import os
from vectors import LorentzVector, Vector3D

In [2]:
data = pd.read_csv("output_events_pion/events_225.35.csv.zip")
print(data)

    ievent  iparticle  truth_energy   pid     px     py       pz        e  \
0        1          0        225.35    14  1.662  0.495  159.278  159.287   
1        1          1        225.35  2212 -0.863 -0.384    7.046    7.171   
2        1          2        225.35   211 -0.278 -0.050    2.981    2.998   
3        1          3        225.35  2212  0.544 -1.155    8.058    8.212   
4        1          4        225.35  -211 -0.213  0.211    2.590    2.611   
5        1          5        225.35   -13 -0.016 -0.058    0.802    0.811   
6        1          6        225.35    14 -0.135 -0.735    5.016    5.071   
7        1          7        225.35    22 -0.010  0.283    1.509    1.535   
8        1          8        225.35    22  0.040  0.354    1.260    1.309   
9        1          9        225.35    22 -0.069 -0.004    0.234    0.244   
10       1         10        225.35    22 -0.308  0.252    1.935    1.976   
11       1         11        225.35    22  0.186  0.495    1.326    1.427   

This function is taken from FORESEE to model the two body decay of $\pi^+$ / $K^+$ to $
\mu^+$ and neutrino. It provides us with the produced particle's four momentum in lab frame


In [3]:
def twobody_decay(p0, m0, m1, m2, phi, costheta):
    """
    Decays p0 -> p1 + p2 and returns the momenta of p1 and p2 in the lab frame.
    """
    zaxis = Vector3D(0, 0, 1)
    rotaxis = zaxis.cross(p0.vector).unit()
    rotangle = zaxis.angle(p0.vector)

    energy1 = (m0 * m0 + m1 * m1 - m2 * m2) / (2. * m0)
    energy2 = (m0 * m0 - m1 * m1 + m2 * m2) / (2. * m0)
    momentum1 = math.sqrt(energy1 * energy1 - m1 * m1)

    px1 = momentum1 * math.sqrt(1. - costheta * costheta) * np.cos(phi)
    py1 = momentum1 * math.sqrt(1. - costheta * costheta) * np.sin(phi)
    pz1 = momentum1 * costheta
    p1_rest = LorentzVector(-px1, -py1, -pz1, energy1)
    p2_rest = LorentzVector(px1, py1, pz1, energy2)

    if rotangle != 0:
        p1_rest = p1_rest.rotate(rotangle, rotaxis)
        p2_rest = p2_rest.rotate(rotangle, rotaxis)

    p1_lab = p1_rest.boost(-1. * p0.boostvector)
    p2_lab = p2_rest.boost(-1. * p0.boostvector)

    return p1_lab, p2_lab



To provide weight to the produced particle, we define the function calculate_decay_probability().The probability of decay of pion is given by $exp(-L/c\tau\times\gamma)$. Here, L is the distance to detector, c$\tau$ is the lifetime of pion and gamma is the boost factor caculated using the formula Energy_pion/mass_pion.

The decay_pion() function is defined to use two_body_decay() function specifically for pion decay. The mass specifications are:

pion_mass = 0.13957  # GeV/c^2


muon_mass = 0.10566  # GeV/c^2

neutrino_mass = 0.0  # GeV/c^2 (assumed negligible)

I have considered L as 10 m. Pion $\tau$ is 2.6033×10−8 seconds and hence we have $c\tau$ = 7.8 m


In [4]:
def calculate_decay_probability(e, m, c_tau, detector_distance=10):
    """
    Calculate the decay probability of a particle using its momentum and lifetime.
    """
    #p = np.sqrt(px**2 + py**2 + pz**2)
    gamma = e / m
    mean_decay_length = c_tau * gamma
    decay_probability = np.exp(-detector_distance / mean_decay_length)
    return decay_probability


def decay_pion(px, py, pz, e):
    """Simulate pion decay into mu+ and neutrino."""
    pion_mass = 0.13957  # GeV/c^2
    muon_mass = 0.10566  # GeV/c^2
    neutrino_mass = 0.0  # GeV/c^2 (assumed negligible)

    # Compute the momentum magnitude of the pion
    momentum_magnitude = np.sqrt(px**2 + py**2 + pz**2)

    # Pion 4-momentum
    p4_pion = LorentzVector(px, py, pz, e)

    # Sample the decay angle phi uniformly
    phi = np.random.uniform(0, 2 * np.pi)

    # Compute cos(theta) as pz / |p|
    costheta = pz / momentum_magnitude if momentum_magnitude > 0 else 0.0

    # Perform the decay
    muon, neutrino = twobody_decay(p4_pion, pion_mass, muon_mass, neutrino_mass, phi, costheta)
    return muon, neutrino


Next, in the events dataframe I introduce first of all the decay products. To take note of the parent, I create a flag that says product for the produced particles, parent for pion and original for all the other particles. I also introduce another column called weight which has entries 1 for all other particles and $exp(-L/c\tau\times\gamma)$ for the products. 

In [5]:
def decay_and_flag(data, detector_distance):
    """Perform pion decay, flag rows, and calculate weights."""
    updated_data = []
    c_tau_pion = 7.8 ###decay width in meter   pion mean lifetime 2.6033×10**−8 a

    for _, row in data.iterrows():
        if row['pid'] == 211:
            decay_probability = calculate_decay_probability(
                row['e'], m=0.13957, c_tau=c_tau_pion, detector_distance=detector_distance
            )

            muon, neutrino = decay_pion(row['px'], row['py'], row['pz'], row['e'])

            parent_row = row.copy()
            parent_row['decay_flag'] = 'parent'
            parent_row['weight'] = 1.0
            updated_data.append(parent_row)

            muon_row = row.copy()
            muon_row['pid'] = -13
            muon_row['px'], muon_row['py'], muon_row['pz'], muon_row['e'] = \
                muon.px, muon.py, muon.pz, muon.e
            muon_row['parent_pid1'] = 211
            muon_row['parent_pid2'] = 90
            muon_row['decay_flag'] = 'product'
            muon_row['weight'] = decay_probability
            updated_data.append(muon_row)

            neutrino_row = row.copy()
            neutrino_row['pid'] = 14
            neutrino_row['px'], neutrino_row['py'], neutrino_row['pz'], neutrino_row['e'] = \
                neutrino.px, neutrino.py, neutrino.pz, neutrino.e
            neutrino_row['parent_pid1'] = 211
            neutrino_row['parent_pid2'] = 90
            neutrino_row['decay_flag'] = 'product'
            neutrino_row['weight'] = decay_probability
            updated_data.append(neutrino_row)
        else:
            row['decay_flag'] = 'original'
            row['weight'] = 1.0
            updated_data.append(row)

    return pd.DataFrame(updated_data)




Here I comapre the input events file to the output after pion_decay

In [6]:
data = pd.read_csv("output_events_pion/events_225.35.csv.zip")
print(data)
decay_and_flag(data, detector_distance=10)

    ievent  iparticle  truth_energy   pid     px     py       pz        e  \
0        1          0        225.35    14  1.662  0.495  159.278  159.287   
1        1          1        225.35  2212 -0.863 -0.384    7.046    7.171   
2        1          2        225.35   211 -0.278 -0.050    2.981    2.998   
3        1          3        225.35  2212  0.544 -1.155    8.058    8.212   
4        1          4        225.35  -211 -0.213  0.211    2.590    2.611   
5        1          5        225.35   -13 -0.016 -0.058    0.802    0.811   
6        1          6        225.35    14 -0.135 -0.735    5.016    5.071   
7        1          7        225.35    22 -0.010  0.283    1.509    1.535   
8        1          8        225.35    22  0.040  0.354    1.260    1.309   
9        1          9        225.35    22 -0.069 -0.004    0.234    0.244   
10       1         10        225.35    22 -0.308  0.252    1.935    1.976   
11       1         11        225.35    22  0.186  0.495    1.326    1.427   

,ievent,iparticle,truth_energy,pid,px,py,pz,e,parent_pid1,parent_pid2,decay_flag,weight
0,1.0,0.0,225.35,14.0,1.662000,0.495000,159.278000,159.287000,14.0,14.0,original,1.000000
1,1.0,1.0,225.35,2212.0,-0.863000,-0.384000,7.046000,7.171000,2214.0,90.0,original,1.000000
2,1.0,2.0,225.35,211.0,-0.278000,-0.050000,2.981000,2.998000,213.0,90.0,parent,1.000000
2,1.0,2.0,225.35,-13.0,-0.153099,-0.028316,1.614478,1.625406,211.0,90.0,product,0.942061
2,1.0,2.0,225.35,14.0,-0.109336,-0.018885,1.199618,1.204738,211.0,90.0,product,0.942061
3,1.0,3.0,225.35,2212.0,0.544000,-1.155000,8.058000,8.212000,2214.0,90.0,original,1.000000
4,1.0,4.0,225.35,-211.0,-0.213000,0.211000,2.590000,2.611000,-4224.0,90.0,original,1.000000
5,1.0,5.0,225.35,-13.0,-0.016000,-0.058000,0.802000,0.811000,421.0,90.0,original,1.000000
6,1.0,6.0,225.35,14.0,-0.135000,-0.735000,5.016000,5.071000,421.0,90.0,original,1.000000
7,1.0,7.0,225.35,22.0,-0.010000,0.283000,1.509000,1.535000,111.0,90.0,original,1.000000


Next, I need to calculate the observables. I have made a few changes to the previous code. First of all, I have put a condition that does not take into account rows having pions. I have also added another column with weight in which I have just multiplied all the probabilities of pion decay $\textbf{I am not sure if multiplying probability  is correct we should have now more number of events since we also have contribution from pion decay?}$



In [7]:
def calculate_observables_with_flags(data):
    """Calculate observables, incorporating the decay flag and weights."""
    events = []
    grouped_data = data.groupby('ievent')

    for ievent, evt in grouped_data:
        #####CONDITION ON PION. 
        evt = evt[evt['decay_flag'] != 'parent']
        ########################
        
        
        e_mu_minus, e_mu_plus, has_charm, e_em, e_visible, ptx, pty, ht = 0, 0, 0, 0, 0, 0, 0, 0
        pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus = 0, 0, 0, 0
        event_weight = 1.0

        for _, row in evt.iterrows():
            truth_energy, pid, parent_pid1, decay_flag, weight = row.iloc[2], row.iloc[3], row.iloc[8], row['decay_flag'], row['weight']
            px, py, pz, e = row.iloc[4], row.iloc[5], row.iloc[6], row.iloc[7]

            if decay_flag == 'product':
                event_weight *= weight

            if abs(parent_pid1) in {411, 421, 431, 4122, 15, 521, 511, 531, 443, 5122, 4232, 4112, 5232}: 
                has_charm = 1

            if pid == 13: 
                if e > e_mu_minus:
                    e_mu_minus = e 
                    pt_mu_minus = np.sqrt(px**2 + py**2) 
                    phi_mu_minus = np.arctan2(py, px)
            elif pid == -13: 
                if e > e_mu_plus:
                    e_mu_plus = e 
                    pt_mu_plus = np.sqrt(px**2 + py**2) 
                    phi_mu_plus = np.arctan2(py, px)

            if abs(pid) in {22, 11}: 
                e_em += e 
            if abs(pid) not in {12, 14, 16, 39}: 
                e_visible += e 
                ptx += px 
                pty += py 
                ht += np.sqrt(px**2 + py**2) 

        pt_mis = np.sqrt(ptx**2 + pty**2)
        phi_vis = np.arctan2(pty, ptx)
        phi_mis = np.arctan2(-pty, -ptx)

        events.append([ievent, truth_energy, e_mu_minus, e_mu_plus, e_em, has_charm, e_visible, pt_mis, ht, pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus, phi_mis, phi_vis, event_weight])

    columns = ['ievent', 'truth_energy', 'e_mu_minus', 'e_mu_plus', 'e_em', 'has_charm', 'e_visible', 'pt_mis', 'ht', 'pt_mu_minus', 'pt_mu_plus', 'phi_mu_minus', 'phi_mu_plus', 'phi_mis', 'phi_vis', 'weight']
    observables = pd.DataFrame(events, columns=columns)

    return observables


In [8]:
energies = [225.35]
processes = ["pion"]

for process in processes: 
    for energy in energies:
        filename = f"output_events_{process}/events_{str(energy)}.csv.zip"

        if not os.path.isfile(filename):
            print(f"File not found: {filename}")
            continue

        print(process, energy)
        data = pd.read_csv(filename)

        if data.empty:
            print(f"Data is empty for {filename}")
            continue

        if 'ievent' not in data.columns:
            print(f"Column 'ievent' not found in {filename}")
            continue

        if 0 not in data.index or np.isnan(data.loc[0, 'ievent']):
            print(f"Fixing DataFrame for {filename}")
            
            data = fix_dataframe(data)
        
        
        data1 = decay_and_flag(data, detector_distance=10.0)
        observables = calculate_observables_with_flags(data1)
        csv_file = f"output_events_{process}/observables_{str(energy)}.csv.zip"
        observables.to_csv(csv_file, index=False, compression='zip')



pion 225.35


In [9]:
data = pd.read_csv("output_events_pion/observables_225.35.csv.zip")
data

,ievent,truth_energy,e_mu_minus,e_mu_plus,e_em,has_charm,e_visible,pt_mis,ht,pt_mu_minus,pt_mu_plus,phi_mu_minus,phi_mu_plus,phi_mis,phi_vis,weight
0,1.0,225.35,0,1.821561,14.599,1,58.812966,1.221419,7.420343,0,0.217879,0,-2.940614,-0.248813,2.892779,0.803477
